# Data Exploration for 3D U-Net Material Learning

This notebook explores the dataset structure and visualizes sample microstructures and stress fields.

In [ ]:
# Import required libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set up plotting
plt.style.use('default')
%matplotlib inline

## 1. Dataset Overview

Let's first understand the structure of our data files.

In [ ]:
# Check if data directory exists
DATA_PATH = Path("../data/data_files/")

if DATA_PATH.exists():
    data_files = list(DATA_PATH.glob("*.pt"))
    print(f"Found {len(data_files)} data files")
    print(f"\nFirst 5 files:")
    for f in data_files[:5]:
        print(f"  {f.name}")
else:
    print(f"⚠️ Data directory not found: {DATA_PATH}")
    print("Please create the data directory and add your data files.")

## 2. Load Sample Data

Load a single data file and examine its structure.

In [ ]:
# Try to load the first input/output pair
try:
    input_file = DATA_PATH / "data_elasticity_3D_128_0_L0_S0_input.pt"
    output_file = DATA_PATH / "data_elasticity_3D_128_0_L0_S0_output.pt"
    
    if input_file.exists() and output_file.exists():
        print("Loading data files...")
        input_data = torch.load(input_file, map_location='cpu')
        output_data = torch.load(output_file, map_location='cpu')
        
        # Extract tensors
        young_modulus = input_data['input'] if 'input' in input_data else input_data
        stress_field = output_data['output'] if 'output' in output_data else output_data
        
        print(f"✅ Data loaded successfully!")
        print(f"\nYoung's Modulus shape: {young_modulus.shape}")
        print(f"Stress Field shape: {stress_field.shape}")
        print(f"\nYoung's Modulus range: [{young_modulus.min():.3f}, {young_modulus.max():.3f}]")
        print(f"Stress Field range: [{stress_field.min():.3f}, {stress_field.max():.3f}]")
    else:
        print(f"⚠️ Data files not found. Please add data files to {DATA_PATH}")
        young_modulus = None
        stress_field = None
        
except Exception as e:
    print(f"❌ Error loading data: {e}")
    young_modulus = None
    stress_field = None

## 3. Visualize a Single Sample

Let's visualize the microstructure (Young's modulus) and corresponding stress field for one sample.

In [ ]:
if young_modulus is not None:
    # Select a sample to visualize
    sample_idx = 0
    slice_idx = 32  # Middle slice
    
    # Extract single sample
    E_sample = young_modulus[sample_idx, 0].numpy()
    stress_sample = stress_field[sample_idx, 0].numpy()
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Young's modulus slices
    im1 = axes[0, 0].imshow(E_sample[:, :, slice_idx].T, cmap='copper', origin='lower')
    axes[0, 0].set_title(f'Young\'s Modulus - XY Slice (z={slice_idx})')
    axes[0, 0].set_xlabel('X')
    axes[0, 0].set_ylabel('Y')
    plt.colorbar(im1, ax=axes[0, 0])
    
    im2 = axes[0, 1].imshow(E_sample[:, slice_idx, :].T, cmap='copper', origin='lower')
    axes[0, 1].set_title(f'Young\'s Modulus - XZ Slice (y={slice_idx})')
    axes[0, 1].set_xlabel('X')
    axes[0, 1].set_ylabel('Z')
    plt.colorbar(im2, ax=axes[0, 1])
    
    im3 = axes[0, 2].imshow(E_sample[slice_idx, :, :].T, cmap='copper', origin='lower')
    axes[0, 2].set_title(f'Young\'s Modulus - YZ Slice (x={slice_idx})')
    axes[0, 2].set_xlabel('Y')
    axes[0, 2].set_ylabel('Z')
    plt.colorbar(im3, ax=axes[0, 2])
    
    # Stress field slices
    im4 = axes[1, 0].imshow(stress_sample[:, :, slice_idx].T, cmap='viridis', origin='lower')
    axes[1, 0].set_title(f'Stress Field σ_xx - XY Slice (z={slice_idx})')
    axes[1, 0].set_xlabel('X')
    axes[1, 0].set_ylabel('Y')
    plt.colorbar(im4, ax=axes[1, 0])
    
    im5 = axes[1, 1].imshow(stress_sample[:, slice_idx, :].T, cmap='viridis', origin='lower')
    axes[1, 1].set_title(f'Stress Field σ_xx - XZ Slice (y={slice_idx})')
    axes[1, 1].set_xlabel('X')
    axes[1, 1].set_ylabel('Z')
    plt.colorbar(im5, ax=axes[1, 1])
    
    im6 = axes[1, 2].imshow(stress_sample[slice_idx, :, :].T, cmap='viridis', origin='lower')
    axes[1, 2].set_title(f'Stress Field σ_xx - YZ Slice (x={slice_idx})')
    axes[1, 2].set_xlabel('Y')
    axes[1, 2].set_ylabel('Z')
    plt.colorbar(im6, ax=axes[1, 2])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Sample {sample_idx} visualized")
    print(f"   You can see the spherical inclusions (darker regions) in the Young's modulus")
    print(f"   And the corresponding stress concentration patterns in the stress field")

## 4. Statistical Analysis

Analyze the distribution of values across multiple samples.

In [ ]:
if young_modulus is not None:
    # Compute statistics
    print("Dataset Statistics:")
    print("=" * 50)
    print(f"\nYoung's Modulus:")
    print(f"  Mean: {young_modulus.mean():.4f}")
    print(f"  Std:  {young_modulus.std():.4f}")
    print(f"  Min:  {young_modulus.min():.4f}")
    print(f"  Max:  {young_modulus.max():.4f}")
    
    print(f"\nStress Field:")
    print(f"  Mean: {stress_field.mean():.4f}")
    print(f"  Std:  {stress_field.std():.4f}")
    print(f"  Min:  {stress_field.min():.4f}")
    print(f"  Max:  {stress_field.max():.4f}")
    
    # Plot histograms
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(young_modulus.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Young\'s Modulus')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Young\'s Modulus Values')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist(stress_field.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black', color='orange')
    axes[1].set_xlabel('Stress σ_xx')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Stress Values')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Using the MaterialDataset Class

Now let's use our custom MaterialDataset class to load data properly.

In [ ]:
# Import our dataset class
import sys
sys.path.insert(0, '../src')

from ann_solid_materials import MaterialDataset

# Create dataset
dataset = MaterialDataset(
    data_path=str(DATA_PATH),
    n_samples=128,
    stress_number=0,  # σ_xx component
    load_number=0,    # Unit strain in x-direction
)

# Load data
try:
    dataset.load_data()
    print(f"✅ Dataset loaded: {len(dataset)} samples")
    
    # Get statistics
    stats = dataset.get_statistics()
    print(f"\nDataset Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value:.6f}")
    
except FileNotFoundError as e:
    print(f"❌ {e}")

## 6. Sample a Batch

Test the dataset by sampling a batch of data.

In [ ]:
if dataset._data_loaded:
    # Get a sample
    input_sample, output_sample = dataset[0]
    
    print(f"Single sample shapes:")
    print(f"  Input:  {input_sample.shape}")
    print(f"  Output: {output_sample.shape}")
    
    # Create a data loader
    from torch.utils.data import DataLoader
    
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    # Get one batch
    batch_input, batch_output = next(iter(loader))
    
    print(f"\nBatch shapes:")
    print(f"  Inputs:  {batch_input.shape}")
    print(f"  Outputs: {batch_output.shape}")
    
    # Visualize the batch
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for i in range(4):
        axes[0, i].imshow(batch_input[i, 0, :, :, 32].T, cmap='copper', origin='lower')
        axes[0, i].set_title(f'Sample {i} - Young\'s Modulus')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(batch_output[i, 0, :, :, 32].T, cmap='viridis', origin='lower')
        axes[1, i].set_title(f'Sample {i} - Stress Field')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Next Steps

Now that we understand the data structure, we can:

1. **Train a model** - See `02_model_training.ipynb`
2. **Evaluate predictions** - See `03_results_visualization.ipynb`
3. **Run hyperparameter search** - See `../hyperparameter_search.py`

Refer to the main README for more information on training and evaluation.